# AutoPINN — Experiment Progress

Reads `results.tsv` and plots the research frontier.
Run this after an overnight session to see what happened.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

df = pd.read_csv('results.tsv', sep='\t')
print(f"Total experiments: {len(df)}")
print(f"Kept: {(df.status == 'keep').sum()}  "
      f"Discarded: {(df.status == 'discard').sum()}  "
      f"Crashed: {(df.status == 'crash').sum()}")
df.tail(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'AutoPINN Progress — {len(df)} Experiments, '
    f'{(df.status == "keep").sum()} Kept Improvements',
    fontsize=13, fontweight='bold'
)

SOTA = {'Convection': 1e-4, 'Allen-Cahn': 3.26e-6}
BASELINE = {'Convection': 1.0, 'Allen-Cahn': 0.5}

for ax, problem in zip(axes, ['Convection', 'Allen-Cahn']):
    sub = df[df.problem == problem].copy().reset_index(drop=True)
    sub['exp_num'] = range(len(sub))

    # valid (non-crash, non-zero) results
    valid = sub[sub.L2_error > 0]
    kept = valid[valid.status == 'keep']
    discarded = valid[valid.status == 'discard']
    crashed = sub[sub.status == 'crash']

    # scatter all experiments
    ax.scatter(discarded.exp_num, discarded.L2_error,
               c='#aaaaaa', s=40, alpha=0.6, zorder=2, label='Discarded')
    ax.scatter(kept.exp_num, kept.L2_error,
               c='#2ecc71', s=60, zorder=4, label='Kept',
               edgecolors='black', linewidths=0.5)
    if len(crashed):
        ax.scatter(crashed.exp_num,
                   [valid.L2_error.max() * 1.1] * len(crashed),
                   c='#e74c3c', s=40, marker='x', zorder=3, label='Crash')

    # running minimum frontier
    if len(kept):
        frontier_x = [0]
        frontier_y = [BASELINE[problem]]
        best = BASELINE[problem]
        for _, row in kept.iterrows():
            if row.L2_error < best:
                best = row.L2_error
                frontier_x.extend([row.exp_num, row.exp_num])
                frontier_y.extend([frontier_y[-1], best])
        frontier_x.append(len(sub) - 1)
        frontier_y.append(best)
        ax.plot(frontier_x, frontier_y, 'b-', lw=1.5,
                alpha=0.7, zorder=3, label='Frontier')

    # SOTA line
    ax.axhline(SOTA[problem], color='red', linestyle='--',
               linewidth=1, alpha=0.8, label=f'SOTA ({SOTA[problem]:.0e})')

    ax.set_yscale('log')
    ax.set_xlabel('Experiment #', fontsize=11)
    ax.set_ylabel('L2 Relative Error (log scale)', fontsize=11)
    ax.set_title(problem, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, which='both')
    ax.yaxis.set_major_formatter(ticker.LogFormatterMathtext())

plt.tight_layout()
plt.savefig('progress.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to progress.png")

In [ ]:
# Best results per problem
print("=== Best results so far ===")
for problem in ['Convection', 'Allen-Cahn']:
    sub = df[(df.problem == problem) & (df.L2_error > 0) & (df.status == 'keep')]
    if len(sub):
        best = sub.loc[sub.L2_error.idxmin()]
        print(f"\n{problem}:")
        print(f"  Best L2:  {best.L2_error:.6f}")
        print(f"  Commit:   {best.commit}")
        print(f"  Change:   {best.description}")
    else:
        print(f"\n{problem}: no improvements yet")

# Top 10 improvements
print("\n=== Top 10 kept experiments ===")
kept = df[(df.status == 'keep') & (df.L2_error > 0)].copy()
kept_sorted = kept.sort_values('L2_error')
print(kept_sorted[['problem', 'L2_error', 'description']].head(10).to_string(index=False))

In [ ]:
# Experiment efficiency: kept rate over time
df_valid = df[df.status != 'crash'].copy().reset_index(drop=True)
df_valid['kept_cumsum'] = (df_valid.status == 'keep').cumsum()
df_valid['keep_rate'] = df_valid['kept_cumsum'] / (df_valid.index + 1)

plt.figure(figsize=(8, 4))
plt.plot(df_valid.index, df_valid.keep_rate * 100, 'b-', lw=2)
plt.xlabel('Experiment #')
plt.ylabel('Cumulative keep rate (%)')
plt.title('Research efficiency over time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()